# Importation of necessary Python Libs

## Maths or data structure-related libraries

In [5]:
import math
import numpy as np
import random
import pandas as pd
from collections import deque, namedtuple
from re import match
from typing import Tuple, List, Set, Dict
from sklearn.linear_model import BayesianRidge

## Libraries for visualization/record tracking

In [6]:
import matplotlib.pyplot as plt
import tqdm

## Libraries for loading data

In [3]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


## Libraries for machine leanring

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer

# Implementation of the VAEQL algoritm

The link to my manuscript: <a>https://www.overleaf.com/project/67cb575babefcc1067d01469</a>

## Defining the miscelaneous methods

### Defining the training method depending on the hardware device

In [8]:
def training_func():
    device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu" # CUDA doesn't work with the AMD GPUs of MacBook M1
    if device == "cuda:0":
        print("Training on the GPU")
    else:
        print("Training on the CPU")

    ENV = namedtuple('env', ('name', 'n_actions', 'encoding_dim'))

### Copy and paste the benchmark dataframe pre-processing methods from the previous experiments

In [9]:
def identify_binary_and_numerical_features(df: pd.DataFrame) -> Tuple[List[str]]:
     # 1. Identify “binary” columns (unique values ⊆ {0,1})
    categorical_cols = [col for col in df.columns if match(r".+-is_.+", col)]

    # 2. All the other numeric columns
    numeric_cols = list(set(df.columns) - set(categorical_cols))

    # return to a tuple of two lists of column names
    return numeric_cols, categorical_cols

In [10]:
def normalize_numerical_features(df: pd.DataFrame, num_features: List[str]) -> pd.DataFrame:
    # Create a MinMaxScaler instance
    scaler = MinMaxScaler()

    # Fit and transform the numerical features to the range [0, 1]
    df[num_features] = scaler.fit_transform(df[num_features])

    return df

In [11]:
def generate_masks_for_missingness(
    original_df: pd.DataFrame,
    amputed_df: pd.DataFrame,
    num_feats: List[str],
    cat_feats: List[str],
    imputed_original_df: pd.DataFrame = None,
) -> Tuple[np.ndarray, np.ndarray]:

    if not type(imputed_original_df) == pd.DataFrame:
        print("No missingness!")
        imputed_original_df = original_df.copy()

    # Check if the dataframes match in shape
    if not original_df.shape == imputed_original_df.shape == amputed_df.shape:
        raise Exception("Sorry, the three dataframes do not match in shape")

    # Check if the dataframes have identical column names
    if not set(original_df.columns) == set(imputed_original_df.columns) == set(amputed_df.columns):
        raise Exception("Sorry, the three dataframes do not match in column names")

    # Initialize the maps for numerical and categorical features
    num_feat_map = np.zeros(original_df[num_feats].shape, dtype=int)
    cat_feat_map = np.zeros(original_df[cat_feats].shape, dtype=int)

    # Generate the missingness map for numerical features
    for i, feat in enumerate(num_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        num_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        num_feat_map[:, i] = np.where(amputed_col.isna(), 2, num_feat_map[:, i])  # 2 if amputed

    # Generate the missingness map for one-hot-encoded categorical features
    for i, feat in enumerate(cat_feats):
        original_col = original_df[feat]
        amputed_col = amputed_df[feat]

        # 1: missing in the original dataset, regardless of whether it is amputed
        cat_feat_map[:, i] = np.where(original_col.isna(), 1, 0)  # 1 if missing originally, otherwise 0
        # 2: existing in the original dataset and amputed
        cat_feat_map[:, i] = np.where(amputed_col.isna(), 2, cat_feat_map[:, i])  # 2 if amputed

    # Return the maps as a tuple of two arrays
    return num_feat_map, cat_feat_map

## Defining the Variational Autoencoder part of the VAEQL algorithm

### The VAE module

In [12]:
class MaskedVAE(nn.Module):
    def __init__(
        self,
        num_features: int,
        latent_dim: int | None = None,
        hidden_layer_sizes: tuple[int, ...] = (128, 64)
    ):
        super().__init__()
        # Determine latent dimension (50% reduction if not specified)
        self.num_features = num_features
        self.latent_dim = latent_dim or (num_features // 2)

        # Build encoder: fully connected layers from num_features → latent_dim
        encoder_layers = []
        in_dim = num_features
        for hidden_dim in hidden_layer_sizes:
            encoder_layers.append(nn.Linear(in_dim, hidden_dim))
            encoder_layers.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        self.encoder = nn.Sequential(*encoder_layers)
        # Map to latent parameters
        self.fc1 = nn.Linear(in_dim, self.latent_dim)  # μ
        self.fc2 = nn.Linear(in_dim, self.latent_dim)  # log(σ²)

        # Build decoder: latent_dim → num_features
        decoder_layers = []
        in_dim = self.latent_dim
        for hidden_dim in reversed(hidden_layer_sizes):
            decoder_layers.append(nn.Linear(in_dim, hidden_dim))
            decoder_layers.append(nn.ReLU(inplace=True))
            in_dim = hidden_dim
        decoder_layers.append(nn.Linear(in_dim, num_features))
        self.decoder = nn.Sequential(*decoder_layers)

    def forward(
        self,
        features: torch.Tensor,
        mask: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Args:
            features: (batch_size, num_features) tensor of amputed+normalized data
            mask:     (batch_size, num_features) numpy-based mask array passed as tensor
                      values in {0,1,2}; only used in loss calculation
                      (0=originally observed and not amputed, 1=originally missing, 2=amputed)
        Returns:
            reconstructed: (batch_size, num_features), i.e., the latent representation
            mu:            (batch_size, latent_dim)
            logvar:        (batch_size, latent_dim)
        """
        # Encode
        hidden = self.encoder(features)
        mu = self.fc1(hidden)
        logvar = self.fc2(hidden)

        # Reparameterization trick
        std = torch.exp(logvar * 0.5)
        eps = torch.randn_like(std)
        z = mu + eps * std

        # Decode
        reconstructed = torch.sigmoid(self.decoder(z))
        return reconstructed, mu, logvar

### The loss function

In [13]:
def masked_vae_loss(
    reconstructed: torch.Tensor,
    original: torch.Tensor,
    mu: torch.Tensor,
    logvar: torch.Tensor,
    mask: torch.Tensor,
    beta: float = 1.0,
    eps: float = 1e-8
) -> torch.Tensor:
    """
    Compute masked VAE loss: reconstruction only over entries with mask==0.
    """
    # Elementwise binary cross-entropy loss
    bce = F.binary_cross_entropy(
        reconstructed, original, reduction='none'
    )
    # Only originally observed entries contribute
    observed = (mask == 0).float()
    masked_bce = bce * observed
    reconstruction_loss = masked_bce.sum() / (observed.sum() + eps)

    # KL divergence between q(z|X) and p(z) = N(0,I)
    kl_div = -0.5 * torch.sum(
        1 + logvar - mu.pow(2) - logvar.exp(),
        dim=1
    ).mean()

    return reconstruction_loss + beta * kl_div

### The VAE training method

In [14]:
def train_masked_vae(
    data_df: pd.DataFrame,
    mask_array: np.ndarray,
    hidden_layer_sizes: tuple[int, ...] = (128, 64),
    latent_dim: int | None = None,
    batch_size: int = 32,
    num_folds: int = 5,
    max_epochs: int = 1000,
    learning_rate: float = 1e-3,
    beta: float = 1.0,
) -> list[float]:
    """
    Train the MaskedVAE on a pandas DataFrame with a numpy mask.

    Args:
        data_df:    pandas.DataFrame of shape (n_samples, num_features)
        mask_array: numpy.ndarray of same shape, values {0,1,2}
        ...         other hyperparameters
    Returns:
        List of final validation losses for each fold.
    """
    # Validate shapes
    if data_df.shape != mask_array.shape:
        raise ValueError(
            f"Input data shape {data_df.shape} must match mask shape {mask_array.shape}"
        )

    # define the training device
    device = torch.device("cuda:0") if torch.cuda.is_available() else "cpu" # CUDA doesn't work with the AMD GPUs of MacBook M1
    print(f"Training on {device} ...")

    # Convert inputs to tensors
    features = torch.from_numpy(data_df.values.astype(np.float32))
    mask = torch.from_numpy(mask_array.astype(np.int64))

    num_samples, num_features = features.shape
    latent_dim = latent_dim or (num_features // 2)

    # Prepare cross-validation
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=42)
    fold_losses: list[float] = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(features), start=1):
        # Initialize model and optimizer
        model = MaskedVAE(
            num_features=num_features,
            latent_dim=latent_dim,
            hidden_layer_sizes=hidden_layer_sizes
        ).to(device)
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

        # Create DataLoaders
        train_ds = TensorDataset(
            features[train_idx].to(device),
            mask[train_idx].to(device)
        )
        val_ds = TensorDataset(
            features[val_idx].to(device),
            mask[val_idx].to(device)
        )
        train_loader = DataLoader(
            train_ds, batch_size=batch_size, shuffle=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=batch_size, shuffle=False
        )

        val_losses: list[float] = []
        # Training loop with early stopping
        for epoch in range(1, max_epochs + 1):
            model.train()
            for batch_features, batch_mask in train_loader:
                optimizer.zero_grad()
                recon, mu, logvar = model(batch_features, batch_mask)
                loss = masked_vae_loss(
                    reconstructed=recon,
                    original=batch_features,
                    mu=mu,
                    logvar=logvar,
                    mask=batch_mask,
                    beta=beta
                )
                loss.backward()
                optimizer.step()

            # Validation
            model.eval()
            total_val_loss = 0.0
            with torch.no_grad():
                for batch_features, batch_mask in val_loader:
                    recon, mu, logvar = model(batch_features, batch_mask)
                    batch_loss = masked_vae_loss(
                        reconstructed=recon,
                        original=batch_features,
                        mu=mu,
                        logvar=logvar,
                        mask=batch_mask,
                        beta=beta
                    )
                    total_val_loss += batch_loss.item() * batch_features.size(0)
            avg_val_loss = total_val_loss / len(val_idx)
            val_losses.append(avg_val_loss)

            # Early stopping on 10-epoch moving average
            if epoch >= 10:
                recent_ma = sum(val_losses[-10:]) / 10
                prev_ma   = sum(val_losses[-11:-1]) / 10
                if recent_ma > prev_ma:
                    print(f"Fold {fold}: early stopping at epoch {epoch}")
                    break

        print(f"Fold {fold} final validation loss: {avg_val_loss:.4f}")
        fold_losses.append(avg_val_loss)

    return fold_losses

### Testing on the toy training method above

In [ ]:
# create the list of tuples for pairing input


#### 1. <code>the Regensburg Pediatric Appendicitis Dataset<code> (large number of missing values, 713 patients)

##### Load and pre-process the data

In [ ]:
prefix = "../data_exploration/"

In [ ]:
test_apd_df_MAR = pd.read_csv(f"{prefix}amputed_datasets/appendicitis_data/MAR_15_perc_1.csv")
test_apd_df_MNAR = pd.read_csv(f"{prefix}amputed_datasets/appendicitis_data/MNAR_20_perc_9.csv")
ref_apd_df = pd.read_csv(f"{prefix}preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv")
imputed_apd_df = pd.read_csv(f"{prefix}preprocessed_datasets/Regensburg_Pediatric_Appendicitis_unimputed.csv")

In [ ]:
feat_m1, feat_c1 = identify_binary_and_numerical_features(ref_apd_df)

In [ ]:
test_apd_MAR_mask_num, test_apd_MAR_mask_cat = generate_masks_for_missingness(
    ref_apd_df,
    test_apd_df_MAR,
    feat_m1,
    feat_c1,
    imputed_original_df = imputed_apd_df
)

In [ ]:
test_apd_MNAR_mask_num, test_apd_MNAR_mask_cat = generate_masks_for_missingness(
    ref_apd_df,
    test_apd_df_MNAR,
    feat_m1,
    feat_c1,
    imputed_original_df = imputed_apd_df
)

##### Apply pre-imputation by BRR to the amputed dataframes

In [ ]:
# 1) set up the imputer to use BayesianRidge
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,          # number of imputation rounds, the default value = 300, switched to 10 here for faster training
    tol=1e-3,             # convergence tolerance
    random_state=42,
)

# 3) fit & transform
imputed_array_MAR = imputer.fit_transform(test_apd_df_MAR)
imputed_array_MNAR = imputer.fit_transform(test_apd_df_MNAR)

# 4) rewrap each Array into a DataFrame
imputed_test_apd_df_MAR = pd.DataFrame(imputed_array_MAR, columns=test_apd_df_MAR.columns, index=test_apd_df_MAR.index)
imputed_test_apd_df_MNAR = pd.DataFrame(imputed_array_MNAR, columns=test_apd_df_MNAR.columns, index=test_apd_df_MNAR.index)

/Users/jiz/miniconda3/envs/imputation_env/lib/python3.10/site-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning: [IterativeImputer] Early stopping criterion not reached.
  warnings.warn(


#### 2. <code>the Heart Failure Clinical Records Dataset</code> (no missing values, 299 patients)

##### Load and pre-process the data

In [ ]:
test_hf_df_MAR = pd.read_csv(f"{prefix}amputed_datasets/heart_failure_clinical_records/MAR_25_perc_8.csv")
test_hf_df_MNAR = pd.read_csv(f"{prefix}amputed_datasets/heart_failure_clinical_records/MNAR_5_perc_2.csv")
ref_hf_df = pd.read_csv(f"{prefix}preprocessed_datasets/Heart_Failure_Clinical_Records_complete.csv")

In [ ]:
feat_m2, feat_c2 = identify_binary_and_numerical_features(test_hf_df_MNAR)

In [ ]:
test_hf_MAR_mask_num, test_hf_MAR_mask_cat = generate_masks_for_missingness(
    ref_hf_df,
    test_hf_df_MAR,
    feat_m2,
    feat_c2
)

No missingness!


In [ ]:
test_hf_MNAR_mask_num, test_hf_MNAR_mask_cat = generate_masks_for_missingness(
    ref_hf_df,
    test_hf_df_MNAR,
    feat_m2,
    feat_c2
)

No missingness!


##### Apply pre-imputation by BRR to the amputed dataframes

In [ ]:
# 1) set up the imputer to use BayesianRidge
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=10,          # number of imputation rounds, the default value = 300, switched to 10 here for faster training
    tol=1e-3,             # convergence tolerance
    random_state=42,
)

# 3) fit & transform
imputed_array_MAR = imputer.fit_transform(test_hf_df_MAR)
imputed_array_MNAR = imputer.fit_transform(test_hf_df_MNAR)

# 4) rewrap each Array into a DataFrame
imputed_test_hf_df_MAR = pd.DataFrame(imputed_array_MAR, columns=test_hf_df_MAR.columns, index=test_hf_df_MAR.index)
imputed_test_hf_df_MNAR = pd.DataFrame(imputed_array_MNAR, columns=test_hf_df_MNAR.columns, index=test_hf_df_MNAR.index)

In [ ]:
# 3) fit & transform
imputed_array_MAR = imputer.fit_transform(test_apd_df_MAR)
imputed_array_MNAR = imputer.fit_transform(test_apd_df_MNAR)

# 4) rewrap each Array into a DataFrame
imputed_test_apd_df_MAR = pd.DataFrame(imputed_array_MAR, columns=test_apd_df_MAR.columns, index=test_apd_df_MAR.index)
imputed_test_apd_df_MNAR = pd.DataFrame(imputed_array_MNAR, columns=test_apd_df_MNAR.columns, index=test_apd_df_MNAR.index)

## Defining the MDP Q-learning part of the VAEQL algorithm

In [ ]:
class MDP_Q_learner():
    def init(self,
            device:str,
            input_df: DataFrame,
            states: namedtuple,
            actions: namedtuple,
            gamma: float=0.9, # discount factor
            alpha = 1e-2, # learning rate
            epsilon=0.2,
            ):
        # externally defined input variables, staying constant
        self.DF = input_df
        self.STATES = states
        self.ACTIONS = actions
        self.DEVICE = device
        # variables that stay constantly updated during Q-learning
        self.state = self.calculate_state() # using self.DF
        self.state_next = None
        # on-policy learning, if it doesn't work, might switch to off-policy learning with replay buffer later
        self.policy_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        # alternative choice
        #self.policy_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        # provisional Target-Q table for off-policy learning
        #self.target_q = torch.rand(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)
        #self.target_q = torch.zeros(len(self.ACTIONS), len(self.STATES)).to(self.DEVICE)

    def calculate_state(self) -> torch.Tensor:
        pass

    def update_state(self) -> None:
        pass

    def calculate_reward(self) -> float:
        pass

    def update_policy_Q_table(self) -> None:
        pass

    def start_new_episode(self, state) -> None:
        self.state = self.calculate_state()
        self.state_next = None
        # in the case of off-policy learning
        #self.target_q = self.policy_q.detach().clone()

    def epsilon_greedy_action(self, state) -> torch.Tensor:
        pass

### Defining the overall data loader, pre-processor and trainer class

# Testing the trainer class above on the five pre-processed datasets